## Problem Statement

---
## ⚠️ Important note on this reconstructed notebook

This notebook was rebuilt from your HTML export and prepared for a smoke test, but the RAG pipeline here depends on **three things that need real internet access**, all blocked in the sandbox I used to prepare this:

1. **The source document.** The real notebook uses a proprietary/copyrighted PDF (a real medical manual). I can't reproduce or use that content, so I've substituted `dummy_medical_manual.pdf` — a short, entirely fictional 2-page placeholder document (fake condition names, fake drug names, clearly non-clinical) so the pipeline has *something* to load/chunk/embed/retrieve against.
2. **The embedding model** (`all-mpnet-base-v2`) — downloads from Hugging Face Hub, which wasn't reachable here.
3. **The LLM** (`Mistral-7B-Instruct`, ~5.9GB quantized) — also downloads from Hugging Face Hub, and is too large to be practical to download/run in this sandbox even if the network allowed it.

**What I verified directly:** the PDF-loading and text-chunking steps (no internet required) — confirmed those run correctly against the dummy PDF (see the "Observations (dummy data)" notes a few cells down).

**What I could not verify:** every Observation cell downstream of the embedding/LLM steps (roughly the rest of the notebook) — those describe the *real* Merck Manual run's outputs (real chunk counts, real generated answers, real timing numbers) and are left as-is from your original notebook, since I have no way to regenerate them here.

**Next step:** run this notebook on Colab or Kaggle (both have full internet access) using either your real manual PDF or the dummy one, then send me the completed `.ipynb` back and I'll rewrite the downstream Observation cells to match your actual output — same process we used for the ReneWind and HelmNet notebooks.
---

The healthcare industry is rapidly evolving, with professionals facing increasing challenges in managing vast volumes of medical data while delivering accurate and timely diagnoses. The need for quick access to comprehensive, reliable, and up-to-date medical knowledge is critical for improving patient outcomes and ensuring informed decision-making in a fast-paced environment.

Healthcare professionals often encounter information overload, struggling to sift through extensive research and data to create accurate diagnoses and treatment plans. This challenge is amplified by the need for efficiency, particularly in emergencies, where time-sensitive decisions are vital. Furthermore, access to trusted, current medical information from renowned manuals and research papers is essential for maintaining high standards of care.

To address these challenges, healthcare centers can focus on integrating systems that streamline access to medical knowledge, provide tools to support quick decision-making, and enhance efficiency. Leveraging centralized knowledge platforms and ensuring healthcare providers have continuous access to reliable resources can significantly improve patient care and operational effectiveness.

### Business Context

Common Questions to Answer

1. Diagnostic Assistance : "What are the common symptoms and treatments for pulmonary embolism?"

2. Drug Information : "Can you provide the trade names of medications used for treating hypertension?"

3. Treatment Plans : "What are the first-line options and alternatives for managing rheumatoid arthritis?"

4. Specialty Knowledge : "What are the diagnostic steps for suspected endocrine disorders?"

5. Critical Care Protocols : "What is the protocol for managing sepsis in a critical care unit?"

### Objective

As an AI specialist, your task is to develop a RAG-based AI solution using renowned medical manuals to address healthcare challenges. The objective is to understand issues like information overload, apply AI techniques to streamline decision-making, analyze its impact on diagnostics and patient outcomes, evaluate its potential to standardize care practices, and create a functional prototype demonstrating its feasibility and effectiveness.

### Data Description

The Merck Manuals are medical references published by the American pharmaceutical company Merck & Co., that cover a wide range of medical topics, including disorders, tests, diagnoses, and drugs. The manuals have been published since 1899, when Merck & Co. was still a subsidiary of the German company Merck.

The manual is provided as a PDF with over 4,000 pages divided into 23 sections.

## Installing and Importing Necessary Libraries and Dependencies

In [ ]:
!pip uninstall -y numpy
!pip install --no-cache-dir --force-reinstall \
    numpy==1.25.2 \
    scipy==1.11.4 \
    pandas==1.5.3 \
    huggingface_hub==0.23.2 \
    tiktoken==0.6.0 \
    pymupdf==1.25.1 \
    langchain==0.1.1 \
    langchain-community==0.0.13 \
    chromadb==0.4.22 \
    torch==2.1.2 \
    torchvision==0.16.2 \
    torchaudio==2.1.2 \
    sentence-transformers==2.3.1

In [ ]:
# Installation for GPU llama-cpp-python
# uncomment and run the following code in case GPU is being used
!CMAKE_ARGS="-DLLAMA_CUBLAS=on" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

# Installation for CPU llama-cpp-python
# uncomment and run the following code in case GPU is not being used
# !CMAKE_ARGS="-DLLAMA_CUBLAS=off" FORCE_CMAKE=1 pip install llama-cpp-python==0.1.85 --force-reinstall --no-cache-dir -q

Note :

- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook .

In [ ]:
import numpy as np
print("NumPy version:", np.__version__)

In [ ]:
import os
os.environ["CHROMA_TELEMETRY"] = "FALSE"

Note :

- After running the above cell, kindly restart the runtime (for Google Colab) or notebook kernel (for Jupyter Notebook), and run all cells sequentially from the next cell.
- On executing the above line of code, you might see a warning regarding package dependencies. This error message can be ignored as the above code ensures that all necessary libraries and their dependencies are maintained to successfully execute the code in this notebook .

In [ ]:
# Libraries for processing dataframes,text
try:
    import numpy as np # Explicitly import numpy first
    import json,os
    import tiktoken
    import pandas as pd

    # Libraries for Loading Data, Chunking, Embedding, and Vector Databases
    from langchain.text_splitter import RecursiveCharacterTextSplitter
    from langchain_community.document_loaders import PyMuPDFLoader
    from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
    from langchain_community.vectorstores import Chroma

    # Libraries for downloading and loading the llm
    from huggingface_hub import hf_hub_download
    from llama_cpp import Llama

except ValueError as e:
    print(f"Error during import: {e}")
    print("This might be due to conflicting package versions, especially numpy.")
except Exception as e:
    print(f"An unexpected error occurred during import: {e}")

## Question Answering using LLM

#### Downloading and Loading the model

In [ ]:
model_name_or_path = "TheBloke/Mistral-7B-Instruct-v0.2-GGUF"
model_basename = "mistral-7b-instruct-v0.2.Q6_K.gguf"

⚠️ **Network note:** the next cell downloads a ~5.9GB quantized Mistral-7B model from Hugging Face Hub (`hf_hub_download`). This requires open internet access to `huggingface.co`, which was not available in the sandboxed environment used to prepare this notebook -- I was not able to run this cell or anything downstream of it here. It should work fine on Colab or Kaggle (both have full internet access), but the download + load will take a while, and Colab's free tier may be tight on RAM/disk for a 7B model even quantized -- a Colab Pro/High-RAM runtime or a smaller quantization (e.g. Q4 instead of Q6) may help if you hit resource limits.

In [ ]:
model_path = hf_hub_download(
    repo_id=model_name_or_path,         # The repo ID on Hugging Face
    filename=model_basename             # The specific model file to download
)

In [ ]:
llm = Llama(
    model_path=model_path,
    n_ctx=2300,
    n_gpu_layers=38,
    n_batch=512
)

#### Response

In [ ]:
def response(query,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    model_output = llm(
      prompt=query,
      max_tokens=max_tokens,
      temperature=temperature,
      top_p=top_p,
      top_k=top_k
    )

    return model_output['choices'][0]['text']

In [ ]:
print(response("What treatment options are available for managing hypertension?"))

Summary The LLM identifies key points well and gives clear, concise answers with short options, but the descriptions are limited due to short token usage.

Observations :

1. Load time is fast (318 ms).
2. Token generation (sampling) is very fast (1032 tokens/sec).
3. Prompt evaluation is slower (39 tokens/sec).
4. Main bottleneck is in evaluation time (18 tokens/sec).
5. Total time for the run is ~7.55 seconds.

Overall: good performance, but evaluation speed could be improved.

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(response(user_input))

Summary

1. The LLM accurately identifies sepsis by analyzing clinical information and recognizing key symptoms and signs .
2. It helps support early diagnosis by detecting patterns indicative of this life-threatening condition.

Note - The LLM identifies key points well and gives clear, concise answers with short options, but the descriptions are limited due to short token usage.

Observation :

1. Load time : Fast at 318 ms.
2. Sampling: Very fast at 1046 tokens/sec.
3. Prompt evaluation: Moderate speed (49 tokens/sec).
4. Evaluation (main generation): Still the slowest part at 19 tokens/sec.
5. Total time: ~7.26 seconds.
6. Overall: Great sampling speed, with evaluation still being the main bottleneck.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(response(user_input_2))

Summary

1. The LLM accurately identifies appendicitis by analyzing clinical data.
2. Appendicitis is a medical condition characterized by inflammation of the appendix, often causing abdominal pain and requiring prompt surgical treatment to prevent complications.

Note - The LLM identifies key points well and gives clear, concise answers with short options, but the descriptions are limited due to short token usage.

Observations

Short Summary of Timings:

1. Load time: 318 ms — fast.
2. Sampling speed: 1034 tokens/sec — very fast.
3. Prompt eval: Improved to 109 tokens/sec — efficient.
4. Eval speed: ~19 tokens/sec — still the slowest phase.
5. Total time: ~7.29 seconds.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(response(user_input_3))

Observations

The LLM accurately identifies alopecia areata by recognizing patterns of sudden, patchy hair loss.

Note : The LLM identifies key points well and gives clear, concise answers with short options, but the descriptions are limited due to short token usage.

Summary

1. Load time: 318 ms — fast.
2. Sampling speed: 1034 tokens/sec — excellent.
3. Prompt eval: Slower at 76.7 tokens/sec.
4. Speed: ~18.9 tokens/sec — still the main bottleneck.
5. Total time: ~7.5 seconds.
6. Overall: Good load and sampling performance. Prompt eval slowed slightly. Token evaluation remains the limiting factor.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(response(user_input_4))

Observations :

The LLM identified a patient who has sustained a physical injury to brain tissue

Note : The LLM identifies key points well and gives clear, concise answers with short options, but the descriptions are limited due to short token usage.

Explaination

1. Load time: 318 ms — fast startup
2. Sampling: 1053 tokens/sec — very fast token generation
3. Prompt eval: 94 tokens/sec — efficient prompt processing
4. Eval time: 19 tokens/sec — main bottleneck during generation
5. Total time: ~7.19 seconds overall

Good overall performance, with evaluation speed still the main area to improve

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and cduring a hiking trip, and what should be considered for their care and recovery?"
print(response(user_input_5))

Observation

The LLM identifies a case where someone has fractured their leg while hiking, recognizing the injury and cause. Note - However, due to the 128-token limit, the response was cut off, causing incomplete or inaccurate identification and reducing the model’s effectiveness in fully understanding the situation.

# Explaination :

1. Load time: 318 ms — quick startup
2. Sampling: 1048 tokens/sec — very fast
3. Prompt evaluation: ~85 tokens/sec — solid speed
4. Evaluation (generation): ~19 tokens/sec — main bottleneck
5. Total time: ~7.22 seconds
6. Overall, the model performs well, with evaluation time still the slowest step.

## Question Answering using LLM with Prompt Engineering

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
query_1_prompt = """
What is the protocol for managing sepsis in critical care? Include initial assessment, immediate treatments (fluids, antibiotics), monitoring, guidelines (SOFA, qSOFA), and key treatment timelines. Please respond clearly with brief bullet points.
"""

print(response(query_1_prompt))

Observations The LLM accurately identified key aspects of sepsis management, including initial assessment and immediate treatment interventions
The LLM identifies key points well and gives clear, concise answers with short options, but the descriptions are limited due to short token usage.

Explaination

1. Load time: 318 ms — fast
2. Sampling: 1051 tokens/sec — very fast
3. Prompt eval: 116 tokens/sec — quite efficient
4. Evaluation: 19 tokens/sec — main bottleneck remains
5. Total time: ~7.39 seconds
6. Prompt evaluation improved significantly here, but token evaluation speed is still the limiting factor

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
query_2_prompt = """
You are a medical expert specializing in gastrointestinal conditions. Please answer the following question clearly and concisely:

Question: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

Your answer should include:
- Typical early and late symptoms
- Diagnostic approach
- Medical management options, if any
- Indications for surgical intervention
- Name and brief description of the surgical procedure (e.g., laparoscopic appendectomy)
- Recovery expectations after surgery

Use bullet points or structured sections for clarity and complete 2-3 sentences.
"""

print(response(query_2_prompt))

Observations

The LLM accurately identified gastrointestinal conditions, including Symptoms  and Diagnostic approach.
Note - The LLM identifies key points well and gives clear, concise answers with short options, but the descriptions are limited due to short token usage.

Summary

1. Load time: 318 ms — quick startup
2. Sampling: 1060 tokens/sec — very fast
3. Prompt eval: 153 tokens/se c — significantly improved
4. Evaluation: 19.5 tokens/sec — still the main bottleneck
5. Total time: ~7.81 seconds
6. Prompt evaluation is much faster here, but evaluation speed remains the key area to optimize.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
query_3_prompt = """
You are a dermatologist with expertise in hair and scalp disorders. Please provide a detailed and structured answer to the following question:

Question: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

Please include:
- Common causes of sudden patchy hair loss (e.g., alopecia areata, infections)
- Diagnostic methods (briefly)
- Treatment options including medical and non-medical approaches
- Expected outcomes and prognosis

Format your answer using bullet points or numbered lists for clarity and complete 2-3 sentences.
"""

print(response(query_3_prompt))

Observation

The LLM provided the cause of sudden patchy hair loss (alopecia areata) but omitted treatment and other important details due to the 128-token limit.

Summary

1. Load time: 318 ms — fast
2. Sampling speed: 1052 tokens/sec — excellent
3. Prompt evaluation: 153 tokens/sec — highly efficient
4. Evaluation speed: 19.3 tokens/sec — still the main bottleneck
5. Total time: ~7.90 seconds

Improvements: Prompt evaluation is consistently fast. Limitation: Token generation (evaluation) remains the slowest stage. The prompt evaluation and overall eval time are a bit higher, likely due to the longer prompt length and model workload.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
query_4_prompt = """
You are a neurologist specializing in brain injuries. Please provide a comprehensive and structured answer to the following question:

Question: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

Your response should cover:
- Immediate emergency management and stabilization
- Medical and surgical treatment options
- Rehabilitation therapies and supportive care
- Potential complications and their management
- Prognosis considerations

Please format the answer using bullet points or sections for clarity.
"""

print(response(query_4_prompt))

# Observations

The LLM detects a physical injury to brain tissue and covers immediate emergency management and medical/surgical treatments,
but it omits rehabilitation, potential complications, and prognosis due to response length limits.

# Summary

1. Load time: 318 ms — fast
2. Sampling: 1058 tokens/sec — very efficient
3. Prompt evaluation: 148 tokens/sec — strong performance
4. Evaluation: 19.5 tokens/sec — still the primary bottleneck
5. Total time: ~7.64 seconds

Good performance overall, especially in prompt processing.
Evaluation speed (~19 tokens/sec) continues to be the limiting factor.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
query_5_prompt="""
You are a medical professional with expertise in emergency and wilderness medicine. Please provide a comprehensive and structured answer to the following question:

**Question:** What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

Your response should include:

- Immediate on-site precautions and first aid
- Techniques for immobilizing the fractured leg in a remote setting
- Safe methods of evacuation or transportation to medical care
- Medical treatment and possible surgical interventions
- Rehabilitation and recovery timeline
- Factors influencing recovery, such as age, health status, and fracture severity

Please format your response using bullet points or clearly labeled sections for clarity.
"""


print(response(query_5_prompt))

# Observations

The LLM detects immediate on-site precautions and first aid for a fractured leg but misses details on immobilization techniques , safe evacuation methods, medical and surgical treatments, rehabilitation timeline, and recovery factors due to token limits.

# Summary

LLaMA Timing Summary:

1. Load time: 318 ms — quick
2. Sampling: 1052 tokens/sec — excellent
3. Prompt evaluation: 146 tokens/sec — strong
4. Evaluation: 18.5 tokens/sec — slowest stage
5. Total time: ~8.36 seconds

Prompt eval and sampling are fast
Evaluation speed dropped slightly, increasing total time — this remains the main bottleneck.

## Data Preparation for RAG

### Loading the Data

In [ ]:
#Libraries for processing dataframes,text
import json,os
import tiktoken
import pandas as pd

#Libraries for Loading Data, Chunking, Embedding, and Vector Databases
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.embeddings.sentence_transformer import SentenceTransformerEmbeddings
from langchain_community.vectorstores import Chroma

In [ ]:
# uncomment and run the below code snippets if the dataset is present in the Google Drive
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
# Point this to wherever your PDF lives:
#  - Colab + Drive:   '/content/drive/MyDrive/medical_diagnosis_manual.pdf'
#  - Kaggle:          '/kaggle/input/<your-dataset-folder>/medical_diagnosis_manual.pdf'
#  - Local/dummy run: 'dummy_medical_manual.pdf' (synthetic placeholder content, safe to share/run --
#                       does NOT contain any real Merck Manual or other copyrighted/clinical text)
manual_pdf_path = "dummy_medical_manual.pdf"  # <-- change this to your real manual's path when available


In [ ]:
pdf_loader = PyMuPDFLoader(manual_pdf_path)

In [ ]:
manual = pdf_loader.load()

### Data Overview

#### Checking the first 5 pages

In [ ]:
for i in range(5):
    print(f"Page Number : {i+1}",end="\n")
    print(manual[i].page_content,end="\n")

# Summary:

The LLM displays the content of the first 5 pages from the document manual, showing the initial steps of data preparation by iterating through the pages and printing their contents sequentially

#### Checking the number of pages

In [ ]:
len(manual)

# Observations (dummy data)

`len(manual)` outputs **2** for the dummy placeholder PDF (`dummy_medical_manual.pdf`) — it's a short, 2-page synthetic document, intentionally much smaller than a real medical manual (the original real Merck Manual PDF loaded as 4,114 pages). This dummy document exists only to smoke-test the PDF-loading and chunking mechanics; swap in your real manual's path (see the data-loading cell above) to get realistic page/chunk counts.

### Data Chunking Summary

In [ ]:
text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    encoding_name='cl100k_base',
    chunk_size=500,         # Typically 500–1000 tokens is a good starting point
    chunk_overlap=50        # Small overlap (e.g., 10–100 tokens) helps maintain context across chunks
)

In [ ]:
document_chunks = pdf_loader.load_and_split(text_splitter)

# Command Summary

This command does two things:

1. Loads the PDF from the path specified earlier using pdf_loader.
2. Splits the content into smaller, overlapping chunks using the text_splitter configured with token-based logic (chunk_size=500, chunk_overlap=50).
3. The result, document_chunks, is a list of text chunks ready for:

Embedding into a vector store.

1. Use in Retrieval-Augmented Generation (RAG) systems for accurate and context-aware responses.

In [ ]:
len(document_chunks)

# Observations (dummy data)

Using a plain character-based `RecursiveCharacterTextSplitter` (chunk_size=500, chunk_overlap=50), the 2-page dummy manual was split into **5 chunks** — verified by actually running this step in a sandboxed environment.

⚠️ Two notes:
1. **Chunk count will differ once you use your real manual** — the original real-data run produced 8,894 chunks from 4,114 pages; this dummy document is intentionally tiny (2 pages) just to confirm the pipeline mechanics work.
2. The notebook's actual code uses `RecursiveCharacterTextSplitter.from_tiktoken_encoder(...)`, which downloads a tokenizer file from `openaipublic.blob.core.windows.net` — **also blocked in this sandbox**, so I used the plain character-based splitter as a stand-in to verify the loading/splitting mechanics still work end-to-end. The tiktoken-based version should work fine on Colab/Kaggle.

In [ ]:
document_chunks[0].page_content

Command Chunks Summary

First Chunk Access: The first chunk’s content can be accessed via document_chunks[0].page_content.

In [ ]:
document_chunks[2].page_content

Command Chunks Summary

Text content of the third chunk : The third chunk’s content can be accessed via document_chunks[2].page_content.

In [ ]:
document_chunks[3].page_content

Command Chunks Summary

Fourth Chunk Access: The Fourth chunk’s content can be accessed via document_chunks[3].page_content.

### Embedding

In [ ]:
import numpy as np
import scipy
print("NumPy version:", np.__version__)
print("SciPy version:", scipy.__version__)
print("Dot product test:", np.dot([1, 2], [3, 4]))

In [ ]:
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("all-mpnet-base-v2")
embeddings = model.encode(["This is a test."])
print(embeddings.shape)

Summary: Sentence Embedding with SentenceTransformers

Import: from sentence_transformers import SentenceTransformer imports the SentenceTransformer class for generating sentence-level embeddings.

Model Loading: model = SentenceTransformer("all-mpnet-base-v2") loads the all-mpnet-base-v2 model, a powerful pretrained model optimized for creating dense sentence embeddings.

Encoding: model.encode(["This is a test."]) converts the input sentence into a high-dimensional vector (embedding), capturing its semantic meaning.

Shape Output: print(embeddings.shape) prints the shape of the resulting embedding array.

Since there’s one sentence, output will be: (1, 768), meaning one embedding vector with 768 dimensions.

⚠️ **Network note:** this cell downloads the `all-mpnet-base-v2` sentence-embedding model from Hugging Face Hub. Same limitation as above -- not reachable from this sandbox, but should work fine on Colab/Kaggle.

In [ ]:
embedding_model = SentenceTransformerEmbeddings(model_name="all-mpnet-base-v2")

Summary:

embedding_model = SentenceTransformerEmbeddings(model_name="all-mpnet-base-v2") initializes an embedding interface that wraps the "all-mpnet-base-v2" SentenceTransformer model, making it compatible with frameworks (like LangChain) for generating vector embeddings of text data.

In [ ]:
embedding_1 = embedding_model.embed_query(document_chunks[0].page_content)
embedding_2 = embedding_model.embed_query(document_chunks[1].page_content)

Summary :

1. embedding_1 = embedding_model.embed_query(document_chunks[0].page_content) generates a vector embedding for the first text chunk.
2. embedding_2 = embedding_model.embed_query(document_chunks[1].page_content) does the same for the second text chunk.

These embeddings numerically r epresent the semantic meaning of each chunk, allowing for tasks like similarity search, clustering, or feeding into retrieval-augmented generation pipelines.

In [ ]:
print("Dimension of the embedding vector ",len(embedding_1))
len(embedding_1)==len(embedding_2)

Summary

1. Each embedding vector has 768 dimensions, which is standard for the all-mpnet-base-v2 model.
2. Both embeddings are consistent in size (True), so you can safely compare or use them in downstream tasks like similarity search or clustering.

In [ ]:
embedding_1,embedding_2

### Vector Database

In [ ]:
import os
os.environ["CHROMA_TELEMETRY"] = "FALSE"

In [ ]:
out_dir = 'medical_db'

if not os.path.exists(out_dir):
  os.makedirs(out_dir)

Summary

1. out_dir = 'medical_db' sets the output directory name where you want to save files or a database.
2. if not os.path.exists(out_dir): checks if this directory already exists.
3. os.makedirs(out_dir) creates the directory if it doesn’t exist, preventing errors when saving files later.

In [ ]:
vectorstore = Chroma.from_documents(
    document_chunks,             # your list of document chunks
    embedding_model,       # your embedding model
    persist_directory=out_dir
)

Summary

1. vectorstore = Chroma.from_documents(...) creates a Chroma vector store by:
2. Input documents: Taking your document_chunks (the split text pieces).
3. Embedding model: Using embedding_model (e.g., SentenceTransformer embeddings) to convert text chunks into vectors.
4. Persistence: Saving the vector store to disk at persist_directory=out_dir (product_db folder) for later reuse.

This allows us to efficiently search, retrieve, and manage your document embeddings for tasks like retrieval-augmented generation (RAG).

In [ ]:
vectorstore = Chroma(persist_directory=out_dir,embedding_function=embedding_model)

Summary

1. vectorstore = Chroma(persist_directory=out_dir, embedding_function=embedding_model)
2. This loads an existing Chroma vector store from the product_db directory (out_dir).
3. It uses the provided embedding_model to handle embedding queries or new documents.
4. Since the data is persisted, it avoids re-embedding everything, enabling fast startup and query.

In [ ]:
vectorstore.embeddings

Summary

vectorstore.embeddings gives you the embedding model or function that the Chroma vector store is using internally.

In [ ]:
vectorstore.similarity_search("What are the symptoms and treatment options for pulmonary embolism?", k=3)

Query Summary

vectorstore.similarity_search(...)

1. It performs a semantic similarity search in your Chroma vector store.
2. The query is: "What are the symptoms and treatment options for pulmonary embolism?""
3. k=3 means it returns the top 3 most relevant document chunks based on the query’s meaning.
4. The search compares the query embedding with stored embeddings to find the closest matches.
5. Useful for retrieving focused information from large document collections in a natural, semantic way.

Summary

1. Our  document chunks include metadata and formatting text such as:
2. 'total_pages': 4114 — indicates total pages in the PDF.
3. 'trapped': '' —  PDF extraction metadata.
4. Document(page_content='Chapter 194.') — shows chapter titles embedded in the chunk text.

Note - because the PDF loader extracts all text content indiscriminately, including headers, footers, and page numbers.

These extra texts can affect embedding quality and retrieval relevance if not cleaned before chunking.

In [ ]:
results = vectorstore.similarity_search(
    "What are the symptoms and treatment options for pulmonary embolism?",
    k=3
)

Explanation:

vectorstore.similarity_search(...)

1. Performs a semantic search on your vector store using the query: "What are the symptoms and treatment options for pulmonary embolism?"
2. k=3 requests the top 3 most relevant document chunks based on semantic similarity to the query.
3. The vector store compares the query embedding with stored document embeddings to retrieve the closest matches.

This is useful for quickly finding relevant information from large text collections.

In [ ]:
# Print each result
for i, doc in enumerate(results, 1):
    print(f"\n--- Result {i} ---\n{doc.page_content}")

Summary

1. for i, doc in enumerate(results, 1):
2. Loops through each document in the results list, numbering them starting at 1.
3. print(f"\n--- Result {i} ---\n{doc.page_content}")
4. Prints a clear header (Result 1, Result 2, etc.) followed by the text content of each retrieved document chunk.

Observations The LLM provides detailed responses that include complete chapter information, summaries, and relevant results extracted from the document.
This enables a comprehensive understanding of the topic by combining contextual chapter details with concise summaries, enhancing the quality and depth of observations.

### Retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type='similarity',
    search_kwargs={'k': 3}  # Return top 3 most relevant chunks
)

In [ ]:
rel_docs = retriever.get_relevant_documents("What are the symptoms and treatment options for pulmonary embolism?")
rel_docs

In [ ]:
model_output = llm(
    "What are the symptoms and treatment options for pulmonary embolism?",
    max_tokens=200,
    temperature=0.7
)

# Summary

temperature=0.7 adds some creativity or variability to the output, balancing between deterministic and diverse responses.

In [ ]:
model_output['choices'][0]['text']

# Explanation:

1. model_output is typically a dictionary containing the model’s response.
2. 'choices' holds a list of possible completions/responses.
3. [0] selects the first response in the list.
4. 'text' extracts the actual generated text string.

### System and User Prompt Template

In [ ]:
qna_system_message = (
    "You are a helpful and knowledgeable medical assistant. "
    "Use the provided context to answer medical questions accurately. "
    "If the answer is not contained in the context, say 'I don't know.'"
)

Explanation:

1. Sets the assistant’s role as a helpful and knowledgeable medical assistant.
2. Instructs it to use only the provided context to answer questions accurately.
3. If the context doesn’t contain the answer, the model should respond with "I don't know." instead of guessing.
4. This helps ensure reliable, context-based answers and prevents hallucinations or fabrications.

In [ ]:
qna_user_message_template = (
    "Context: {context}\n\n"
    "Question: {question}\n\n"
    "Answer:"
)

Explanation

1. This defines a template string for formatting user questions in a Q&A system, where:
2. {context} will be replaced with relevant background or retrieved information.
3. {question} will be replaced with the user’s actual question.
4. The template ends with "Answer:" to prompt the model to generate a response based on the given context and question.

### Response Function

In [ ]:
import html

def generate_rag_response(user_input, k=3, max_tokens=128, temperature=0, top_p=0.95, top_k=50):
    global qna_system_message

    relevant_chunks = retriever.get_relevant_documents(query=user_input, k=k)
    context_list = [d.page_content.strip() for d in relevant_chunks]
    context_for_query = " ".join(context_list)

    if len(context_for_query) > 3000:
        context_for_query = context_for_query[:3000] + "..."

    user_prompt = f"""
You are a helpful assistant answering questions based only on the following context.

Context:
{context_for_query}

Question:
{user_input}

Instructions:
- First, provide a brief description (2–3 sentences).
- Then, list 3 to 5 key points in a numbered list (1–5).
- Be clear and concise. Only use the context provided.
"""

    full_prompt = qna_system_message + "\n" + user_prompt

    try:
        response_obj = llm(
            prompt=full_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k
        )

        raw_text = response_obj['choices'][0]['text'].strip()

        # Optional: escape HTML-sensitive characters
        safe_text = html.escape(raw_text)

        # Convert newlines to <br> for HTML display
        html_response = safe_text.replace('\n', '<br>')

        # Wrap in a container for rendering
        return f"<div><strong>Answer:</strong><br>{html_response}</div>"

    except Exception as e:
        return f"<div style='color:red;'>Sorry, an error occurred:<br>{html.escape(str(e))}</div>"

## Question Answering using RAG

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
import re

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(user_input,top_k=2))

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(user_input,top_k=20))

Summary

1. Load time: 400.46 ms — Time to load the model into memory.
2. Sample time: 125.36 ms for 128 tokens (0.98 ms/token) — Time spent generating tokens during sampling.
3. Prompt eval time: 6511.65 ms for 1036 tokens (6.29 ms/token) — Time taken to evaluate the prompt input.
4. Eval time: 7177.86 ms for 127 tokens (56.52 ms/token) — Time spent on the model generating output tokens.
5. Total time: 14037.33 ms (~14 seconds) — Overall time from start to finish for the generation process

Explanation:

1. generate_rag_response is a function that generates a Response using a Retrieval-Augmented Generation (RAG) approach.
2. user_input is the question or query from the user.
3. top_k=20 means the function retrieves the top 20 most relevant documents or chunks from the vector store to provide context for the answer.
4. The retrieved documents are used to inform or augment the LLM’s generation, improving accuracy and grounding in source material

Observation:

1. LLaMA provided a clear and well-structured summary, listing key points effectively and detailing the causes comprehensively.
2. This demonstrates its ability to generate organized and informative content.

Note - LLaMA are producing the same response, despite different top_k values. This likely reflects a limitation in how the LLaMA model handles summarization or context input.

# Trying Chnaging the Prompt with top_k=2 and top_k=20

In [ ]:
user_input_1 = """Summarize the protocol for managing sepsiss in critical care using the 2 documents below.

Focus on:
- Monitoring
- Medications
- Interventions"""
print(generate_rag_response(user_input_1, top_k=2))

In [ ]:
user_input_1 = """Summarize the protocol for managing sepsis in critical care using all 20 documents below.

Focus on:
- Common practices
- Variations
- Step-by-step actions"""
print(generate_rag_response(user_input_1, top_k=2))

# Observations top_k=2 and top_k=2

1. Using top_k=2 retrieves fewer documents, so the model generates a shorter, more generic summary.
The response may be less clear and sometimes cut off, making it hard to identify specific info from the documents.
2. Using top_k=20 provides more source material, but without careful prompt design or chunk handling, the model might still produce similar or overloaded responses.
The summary can improve but might also be less focused if the model can’t fully process all input.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response(user_input_2))

Summary

LLaMA Response to Appendicitis Query

When using generate_rag_response with the prompt "What are the common symptoms for appendicitis?", the LLaMA model:

1. Successfully identified the cause and provided an informative, relevant response.
2. However, the output was shortened, missing some expected symptom details or ending abruptly.

This suggests the need to adjust parameters like max_tokens or improve prompt/context handling to ensure complete and thorough answers.

Observations

LLaMA Timing Overview:

1. Load time: 400.46 ms — Model loading duration.
2. Sample time: 121.87 ms for 128 tokens (~0.95 ms/token) — Token generation speed during sampling phase.
3. Prompt eval time: 6527.40 ms for 1005 tokens (~6.49 ms/token) — Time spent processing the input prompt.
4. Eval time: 6900.59 ms for 127 tokens (~54.34 ms/token) — Time taken for output token generation.
5. Total time: 13765.53 ms (~13.8 seconds) — Complete processing time.

In [ ]:
print(generate_rag_response(user_input_2, top_k=20, temperature=0.5))

# Explaination :

Generates an LLM response using Retrieval-Augmented Generation (RAG).

Parameters:

1. user_input_2
→ The user's query (e.g., a medical or technical question).
2. top_k=20
→ The top 20 most semantically relevant document chunks are retrieved from the vector store to provide context for answering the query.
3. temperature=0.5
→ Controls creativity in the LLM’s response.
A lower value (like 0.5) ensures the response is more focused, factual, and consistent.

The function will:

1. Search the vector store for top 20 relevant pieces of context.
2. Format a context-aware prompt using user_input_2.
3. Pass it to the LLaMA model to generate a reliable answer with controlled variation.

# Observations

1. LLaMA successfully used the top 20 retrieved context chunks to ground its answer, ensuring the response was based on factual content.
2. The model generated a concise and informative response, especially focusing on the cause or high-level summary of the topic (e.g., symptoms or definition).
3. In some cases, the model shortened the text, likely due to max_tokens constraints or early stopping, leading to missing details (e.g., treatment options, complications, etc.).
4. The use of temperature=0.5 helped maintain a focused and stable tone, reducing randomness while still allowing some variation in phrasing.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response(user_input_3))

# Timing Summary

1. Load time: 318.32 ms — time to load the model into memory
2. Sample time: 121.00 ms for 128 tokens (~0.95 ms per token) — token sampling speed, very fast (~1058 tokens/sec)
3. Prompt eval time: 5857.00 ms for 917 tokens (~6.39 ms per token) — processing the input prompt, moderately fast (~157 tokens/sec)
4. Eval time: 7098.69 ms for 127 tokens (~55.90 ms per token) — generating output tokens, slower generation speed (~18 tokens/sec)
5. Total time: 13284.51 ms (~13.28 seconds) — total elapsed time for the entire process

Summary:

1. Prompt evaluation is the main time consumer due to input size (917 tokens).
2. Output generation is slower per token but involves fewer tokens (127 tokens).
3. Sampling is very efficient and negligible in total time.

# Observations

LLaMA What are the effective treatments or solutions for addressing sudden patchy hair loss

When using generate_rag_response with the prompt "What are the effective treatments or solutions for addressing sudden patchy hair loss ....", the LLaMA model:

1. Successfully identified the cause and provided an informative, relevant response.
2. However, the output was shortened, missing some expected details or ending abruptly.

This suggests the need to adjust parameters like max_tokens or improve prompt/context handling to ensure complete and thorough answers.

### Query 4:  What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(user_input_4))

# Observations

Interpretation:

1. Efficient Sampling: Fast output generation per token.
2. Prefix Match: Improves performance if similar prompts are reused.
3. Slow Eval Time per Token: Consistent with LLaMA’s performance, especially for longer outputs (common in inference settings).
4. Overall Latency: ~12–13 seconds per query due to prompt length and model complexity.

# Summary

LLaMA What treatments are recommended for a person who has sustained a physical injury to brain tissue

When using generate_rag_response with the prompt "What treatments are recommended for a person who has sustained a physical injury to brain tissue ....", the LLaMA model:

1. Successfully identified the cause and provided an informative, relevant response.
2. However, the output was shortened, missing some expected details or ending abruptly.

This suggests the need to adjust parameters like max_tokens or improve prompt/context handling to ensure complete and thorough answers.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response(user_input_5))

# Explaination

1. Fast sampling rate for token generation (expected in optimized inference).
2. Prompt evaluation is substantial (~5.7s) due to a moderately large context (909 tokens).
3. Token generation (eval time) is slow per token (~54 ms), typical for LLaMA in CPU or non-optimized environments.
4. Total latency (~13s) is acceptable for large-model inference but may be improved with:
5. Reducing prompt size (e.g., limit top_k context),
6. Increasing system performance (e.g., use GPU or quantized model),
7. Adjusting max_tokens to prevent overgeneration.

# Summary

LLaMA What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

When using generate_rag_response with the prompt "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

1. Successfully identified the cause and provided an informative, relevant response.
2. However, the output was shortened, missing some expected details or ending abruptly.

This suggests the need to adjust parameters like max_tokens or improve prompt/context handling to ensure complete and thorough answers.

### Fine-tuning

In [ ]:
user_input = "What is the protocol for managing sepsis in a critical care unit?"
print(generate_rag_response(user_input,temperature=0.5))

# Explaination

Effect of temperature=0.5 in generate_rag_response

Balanced Output:

Setting temperature=0.5 provides a middle ground between creativity and consistency. The LLM generates responses that are:

Factually reliable

Less repetitive

Still somewhat flexible in phrasing

Use Case Fit:

Ideal for knowledge-based queries, such as medical, technical, or instructional questions, where:

Precision matters

Some variation in sentence structure is acceptable

Stability:

Responses are less random than with higher temperatures (e.g., 0.7+), reducing the risk of hallucinated or overly verbose content.

Observations

Clear & Structured:

The LLM outputs well-organized responses detailing medical conditions, including symptoms, causes, and treatment steps.

Concise but Informative:

Answers provide essential information without unnecessary verbosity, suitable for quick understanding and clinical use.

Stepwise Guidance:

Treatment or management protocols are presented in clear, sequential steps, making them easy to follow.

Accurate Context Use:

Responses are grounded in retrieved medical documents, ensuring relevance and reducing errors.

Balanced Creativity:

Temperature 0.5 helps maintain factual correctness while allowing natural language phrasing for readability.

In [ ]:
user_input_2 = "What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?"
print(generate_rag_response(user_input_2, temperature=0.5))

LLaMA Timing Summary

1. Load time: 318.32 ms — time to load the model into memory
2. Sample time: 108.14 ms for 128 tokens (~0.84 ms per token) — token sampling speed, very fast (~1184 tokens/sec)
3. Prompt eval time: 6626.35 ms for 1005 tokens (~6.59 ms per token) — processing the input prompt, moderately fast (~152 tokens/sec)
4. Eval time: 7107.71 ms for 127 token s (~55.97 ms per token) — generating output tokens, slower generation speed (~18 tokens/sec)
5. Total time: 14051.98 ms (~14.05 seconds) — total elapsed time for the entire process

# Explaination

Effect of temperature=0.5 in generate_rag_response

Balanced Output:

Setting temperature=0.5 provides a middle ground between creativity and consistency. The LLM generates responses that are:

Factually reliable

Less repetitive

Still somewhat flexible in phrasing

Use Case Fit:

Ideal for knowledge-based queries, such as medical, technical, or instructional questions, where:

Precision matters

Some variation in sentence structure is acceptable

Stability:

Responses are less random than with higher temperatures (e.g., 0.7+), reducing the risk of hallucinated or overly verbose content.

Summary

Clear & Structured:

The LLM outputs well-organized responses detailing medical conditions, including symptoms, causes, and treatment steps.

Concise but Informative:

Answers provide essential information without unnecessary verbosity, suitable for quick understanding and clinical use.

In [ ]:
user_input_3 = "What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?"
print(generate_rag_response(user_input_3, temperature=0.5))

LLaMA Timing Summary

Timing Summary

1. Load time: 318.32 ms — time to load the model into memory
2. Sample time: 128.56 ms for 128 tokens (~1.00 ms per token, ~996 tokens/sec) — token sampling speed
3. Prompt eval time: 5,841.75 ms for 917 tokens (~6.37 ms per token, ~157 tokens/sec) — processing input prompt
4. Eval time : 6,968.53 ms for 127 tokens (~54.87 ms per token, ~18 tokens/sec) — generating output tokens
5. Total time: 13,403.78 ms (~13.4 seconds) — overall elapsed time

Effect of temperature=0.5 in generate_rag_response

Balanced Output:

Produces factually reliable, concise responses with moderate flexibility in phrasing.

Best For:

Medical, technical, and instructional queries where precision matters but some variation is acceptable.

Stable Results:

Less randomness than higher temperatures, reducing errors and unnecessary verbosity.

Summary

Clear & Structured:

The LLM outputs well-organized responses detailing medical conditions, including symptoms, causes, and treatment steps.

Concise but Informative:

Answers provide essential information without unnecessary verbosity, suitable for quick understanding and clinical use.

In [ ]:
user_input_4 = "What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?"
print(generate_rag_response(user_input_4, temperature=0.5))

LLaMA Inference Timing Summary

1. Load time: 318.32 ms — time to load the model into memory
2. Sample time: 107.90 ms for 128 tokens (~0.84 ms per token, ~1186 tokens/sec) — very fast token sampling
3. Prompt eval time: 5,449.11 ms for 846 tokens (~6.44 ms per token, ~155 tokens/sec) — processing the input prompt and context
4. Eval time: 7,042.43 ms for 127 tokens (~55.45 ms per token, ~18 tokens/sec) — generating output tokens
5. Total time: 12,809.12 ms (~12.8 seconds) — full end-to-end processing

Summary

1. Prompt evaluation is the most time-consuming step due to relatively long input length.
2. Output generation speed is moderate, typical for LLM token generation.
3. Sampling time is minimal and not a bottleneck.
4. Total latency of ~12.8 seconds can be improved by reducing prompt length or optimizing hardware.

Observations

Clear & Structured:

The LLM outputs well-organized responses detailing medical conditions, including symptoms, causes, and treatment steps.

Concise but Informative:

Answers provide essential information without unnecessary verbosity, suitable for quick understanding and clinical use.

In [ ]:
user_input_5 = "What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?"
print(generate_rag_response(user_input_5, temperature=0.5))

LLaMA Inference Timing Summary

1. Load time: 318.32 ms — model load into memory
2. Sample time: 107.82 ms for 128 tokens (~0.84 ms per token) — very fast (~1187 tokens/sec)
3. Prompt eval time: 5,984.44 ms for 909 tokens (~6.58 ms per token) — input processing speed ~152 tokens/sec
4. Eval time: 7,137.15 ms for 127 tokens (~56.20 ms per token) — output generation speed ~17.8 tokens/sec
5. Total time: 13,446.98 ms (~13.45 seconds)

Observation

Clear & Structured: The LLM generates well-organized responses that break down medical conditions into key components—symptoms, causes, and treatment steps—making it easy to follow and clinically relevant.

Concise & Informative: Responses avoid unnecessary complexity while still covering critical medical insights, making them ideal for quick understanding, triage support, or integration into clinical decision tools.

Recommendations for Production-Ready Usage

1. Limit prompt length to reduce prompt eval time.
2. Limit output tokens to balance detail and latency.
3. Use domain-specific fine-tuning for better relevance.
4. Incorporate manual review or clinician feedback loops.
5. Optimize hardware (GPUs, quantization) for speed improvements.

## Output Evaluation

Let us now use the LLM-as-a-judge method to check the quality of the RAG system on two parameters - retrieval and generation. We illustrate this evaluation based on the answeres generated to the question from the previous section.

- We are using the same Mistral model for evaluation, so basically here the llm is rating itself on how well he has performed in the task.

In [ ]:
groundedness_rater_system_message = (
    "You are an expert evaluator. Your task is to rate the groundedness of an AI-generated answer "
    "based on the provided context. Groundedness means how well the answer is supported by the facts "
    "and information present in the context. Do not consider any information outside the context. "
    "Provide a rating from 1 to 5, where 1 means not grounded at all, and 5 means fully grounded."
)

In [ ]:
relevance_rater_system_message = (
    "You are an expert evaluator. Your task is to rate how relevant an AI-generated answer is to the user's question. "
    "Provide a rating from 1 to 5, where 1 means not relevant at all, and 5 means highly relevant. "
    "Consider only the question and the answer, ignoring any other context."
)

In [ ]:
user_message_template = """
### Question:
{question}

### Context:
{context}

### Answer:
{answer}

Please rate the relevance of the above answer to the question on a scale from 1 (not relevant) to 5 (highly relevant), and briefly explain your rating.
"""

In [ ]:
def generate_ground_relevance_response(user_input,k=3,max_tokens=128,temperature=0,top_p=0.95,top_k=50):
    global qna_system_message,qna_user_message_template
    # Retrieve relevant document chunks
    relevant_document_chunks = retriever.get_relevant_documents(query=user_input,k=3)
    context_list = [d.page_content for d in relevant_document_chunks]
    context_for_query = ". ".join(context_list)

    # Combine user_prompt and system_message to create the prompt
    prompt = f"""[INST]{qna_system_message}\n
                {'user'}: {qna_user_message_template.format(context=context_for_query, question=user_input)}
                [/INST]"""

    response = llm(
            prompt=prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    answer =  response["choices"][0]["text"]

    # Combine user_prompt and system_message to create the prompt
    groundedness_prompt = f"""[INST]{groundedness_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    # Combine user_prompt and system_message to create the prompt
    relevance_prompt = f"""[INST]{relevance_rater_system_message}\n
                {'user'}: {user_message_template.format(context=context_for_query, question=user_input, answer=answer)}
                [/INST]"""

    response_1 = llm(
            prompt=groundedness_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    response_2 = llm(
            prompt=relevance_prompt,
            max_tokens=max_tokens,
            temperature=temperature,
            top_p=top_p,
            top_k=top_k,
            stop=['INST'],
            )

    return response_1['choices'][0]['text'],response_2['choices'][0]['text']

### Query 1: What is the protocol for managing sepsis in a critical care unit?

In [ ]:
ground,rel = generate_ground_relevance_response(user_input="What is the protocol for managing sepsis in a critical care unit?",max_tokens=370)

print(ground,end="\n\n")
print(rel)

LLaMA Inference Timing Summary

Summary:

1. Prompt evaluation dominates total latency due to very long inputs (1,795 and 2,032 tokens).
2. Output generation speed is steady but slower compared to sampling (~60 ms per token).
3. Sampling time is negligible in comparison.
4. Total time ranges from about 22 to 26 seconds, which is considerable for critical environments.
5. Relevance rating: Model provides a relevance score of 5 (high relevance) for answers generated.

Important: Always complement automated ratings with manual clinical review to avoid risks of incomplete or misleading conclusions.

Observation

1. Relevance: The response directly answers the user’s question about appendicitis symptoms and treatment.
2. Completeness: It covers common symptoms, standard surgical treatment, and adjunct antibiotic use, giving a clear clinical picture.
3. Clarity: The explanation is straightforward and easy to understand.
4. Usefulness: It provides practical info relevant for patients or clinicians seeking a quick overview.

### Query 2: What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?

In [ ]:
ground, rel = generate_ground_relevance_response(
    user_input="What are the common symptoms for appendicitis, and can it be cured via medicine? If not, what surgical procedure should be followed to treat it?",
    max_tokens=370
)
print(ground,end="\n\n")
print(rel)

Summary:

1. Prompt evaluation is the most time-consuming step due to relatively long input tokens (1,113 and 1,304 tokens).
2. Output generation speed is steady, around 56–57 ms per token (~17–18 tokens/sec).
3. Sampling time is minimal and efficient.
4. Total time is about 13 seconds for both runs.
5. Relevance rating: The model rates the answer relevance as 5 (high relevance).

Note: Despite high automated relevance scores, manual review is recommended to ensure clinical accuracy and applicability.

Observations

1. Relevance : The response directly answers the user’s question about appendicitis symptoms and treatment.
2. Completeness : It covers common symptoms, standard surgical treatment, and adjunct antibiotic use, giving a clear clinical picture.
3. Clarity : The explanation is straightforward and easy to understand.
4. Usefulness : It provides practical info relevant for patients or clinicians seeking a quick overview.

Note: Despite high automated relevance scores, manual review is recommended to ensure clinical accuracy and applicability.

### Query 3: What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?

In [ ]:
ground, rel = generate_ground_relevance_response(
    user_input="What are the effective treatments or solutions for addressing sudden patchy hair loss, commonly seen as localized bald spots on the scalp, and what could be the possible causes behind it?",
    max_tokens=370
)

print(ground, end="\n\n")
print(rel)

Summary:

1. Prompt evaluation is consistently the major time consumer, especially with longer inputs (1300–1764 tokens).
2. Output token generation speed varies but remains around 58–59 ms per token (~17 tokens/sec).
3. Sampling time is consistently minimal and very efficient (~0.94 ms per token).
4. Total processing time ranges from ~18.8 to ~30.7 seconds, largely dependent on input and output token counts.

Recommendations:

1. Further optimize input prompt length for faster prompt evaluation.
2. Limit output tokens to reduce evaluation time without sacrificing answer completeness.
3. Consider hardware acceleration or model quantization to improve overall latency.

Observation

1. Relevance : The response directly answers the user’s question about appendicitis symptoms and treatment.
2. Completeness : It covers common symptoms, standard surgical treatment, and adjunct antibiotic use, giving a clear clinical picture.
3. Clarity : The explanation is straightforward and easy to understand.
4. Usefulness : It provides practical info relevant for patients or clinicians seeking a quick overview.
5. Relevance rating: The LLM rates the relevance of answers to the questions as 5 (high relevance), indicating generally good alignment with the queries.

Important: Despite high automated relevance scores, manual reevaluation is essential to confirm accuracy and applicability in clinical contexts before making critical decisions.

### Query 4: What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?

In [ ]:
ground, rel = generate_ground_relevance_response(
    user_input="What treatments are recommended for a person who has sustained a physical injury to brain tissue, resulting in temporary or permanent impairment of brain function?",
    max_tokens=370
)

print(ground, end="\n\n")
print(rel)

Summary:

1. Prompt evaluation is the largest chunk of time but here it's relatively faster than previous logs (~6.15–6.45 ms per token).
2. Output generation (eval time) stays consistent around 54–55 ms per token.
3. Sampling remains very efficient and negligible.
4. Total time for these runs is roughly ~13 seconds, significantly faster than the previous large token runs.
5. Token throughput during prompt eval has increased (~155-163 tokens/sec), indicating possible hardware or model size improvements.

Recommendations:

1. Prompt length optimization can still reduce overall latency.
2. Output length control is important to keep generation time manageable.
3. Efficient sampling and stable eval times indicate good model pipeline performance.

Observation

1. Relevance : The response directly answers the user’s question about appendicitis symptoms and treatment.
2. Completeness : It covers common symptoms, standard surgical treatment, and adjunct antibiotic use, giving a clear clinical picture.
3. Clarity : The explanation is straightforward and easy to understand.
4. Usefulness : It provides practical info relevant for patients or clinicians seeking a quick overview.

Important: Despite high automated relevance scores, manual reevaluation is essential to confirm accuracy and applicability in clinical contexts before making critical decisions.

### Query 5: What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?

In [ ]:
ground, rel = generate_ground_relevance_response(
    user_input="What are the necessary precautions and treatment steps for a person who has fractured their leg during a hiking trip, and what should be considered for their care and recovery?",
    max_tokens=370
)

print(ground, end="\n\n")
print(rel)

Detailed timing breakdown for this run:

Run 1:

1. Load time: 318.32 ms — time to load the model into memory
2. Sample time: 285.28 ms for 302 tokens (~0.94 ms per token) — token sampling speed, very fast (~1058 tokens/sec)
3. Prompt eval time: 12,958.26 ms for 1841 tokens (~7.04 ms per token) — processing the input prompt, relatively slower (~142 tokens/sec)
4. Eval time: 17,979.65 ms for 301 tokens (~59.73 ms per token) — generating output tokens, slower generation speed (~16 tokens/sec)
5. Total time: 31,755.42 ms (~31.76 seconds) — total elapsed time for the entire process

Run 2:

1. Load time: 318.32 ms — time to load the model into memory
2. Sample time: 54.65 ms for 58 tokens (~0.94 ms per token) — token sampling speed, very fast (~1061 tokens/sec)
3. Prompt eval time: 16,731.23 ms for 2236 tokens (~7.48 ms per token) — processing the input prompt, relatively slower (~134 tokens/sec)
4. Eval time: 3,476.18 ms for 57 tokens (~60.99 ms per token) — generating output tokens, slower generation speed (~16 tokens/sec)
5. Total time: 20,361.95 ms (~20.36 seconds) — total elapsed time for the entire process

Summary:

1. Prompt evaluation is the most time-consuming step due to large input sizes (1841 and 2236 tokens), taking about 7 ms per token.
2. Output generation is slower per token (~60 ms/token) but overall faster than prompt evaluation because fewer tokens are generated.
3. Sampling time is negligible, showing efficient token selection.
4. Total time is driven mainly by input length and output token count.

Recommendations to Optimize Performance:

1. Reduce prompt length where possible to decrease prompt evaluation time.
2. Limit output tokens to reduce generation latency.
3. Use faster hardware or quantized models to speed up prompt evaluation and output generation

Observations

1. Relevance : The response directly answers the user’s question about appendicitis symptoms and treatment.
2. Completeness : It covers common symptoms, standard surgical treatment, and adjunct antibiotic use, giving a clear clinical picture.
3. Clarity : The explanation is straightforward and easy to understand.
4. Usefulness : It provides practical info relevant for patients or clinicians seeking a quick overview.

## Actionable Insights and Business Recommendations

# Technical Evaluation & Business Recommendations for Your LLM-powered Medical QA System

Technical Evaluation

Impact of top_k on Response Quality:

1. Using top_k=2 returns fewer documents, often leading to shorter, more generic, and sometimes incomplete summaries. This makes it difficult to clearly identify detailed information or nuances from the source documents.
2. Increasing to top_k=20 brings in more data, potentially enabling more comprehensive summaries. However, without careful prompt engineering and chunk management, the model might still produce similar or unfocused outputs because it can’t fully process or integrate all retrieved content due to context length limitations.
3. Variability in response quality also depends on how documents are selected and ranked before input, and on the model’s ability to synthesize rather than merely concatenate data.

LLaMA Model Limitations in Medical Summarization:

1. LLaMA may truncate responses or produce generic outputs when faced with lengthy or complex input.
2. Relying solely on a single prompt or retrieval setting risks generating incomplete or inaccurate medical advice, which could lead to false or misleading clinical interpretations.
3. Multiple retrieval configurations and prompt variations, combined with manual expert review, are necessary to ensure reliability and thoroughness in clinical decision support.

Latency and Token Processing Insights:

1. Prompt evaluation time scales sharply with input length, so longer prompts (like those incorporating 20 documents) increase latency.
2. Token generation speed is moderate; optimizing response length helps reduce overall response time.
3. Sampling time is minimal, indicating efficient token selection during generation.
4. Hardware and model size significantly affect throughput; investing in accelerated hardware can improve performance.

Effect of Temperature Settings:

1. Lower temperatures (0.3-0.5) tend to produce concise, accurate, and reliable outputs suited for clinical applications.
2. Higher temperatures increase creativity and length but may reduce factual precision.

# Business Recommendations

1. Leverage High-Quality Contextual Answers for Clinical Decision Support:
Provide clinicians with grounded, evidence-backed answers to reduce literature search time and increase confidence in decision-making.
2. Implement Continuous Evaluation & Feedback Loops:
Use relevance and groundedness metrics alongside clinician feedback to iteratively improve model accuracy and trustworthiness.
3. Optimize Infrastructure for Scalability and Low Latency:
Given processing times, invest in GPUs or specialized inference hardware to meet real-time needs in critical care.
4. Develop Specialized Medical Modules:
Fine-tune models for domains like sepsis or cardiology to enhance precision and relevance.
5. Design User-Friendly Interfaces:
Present answers with confidence scores and source citations to boost clinician trust and facilitate validation.
6. Ensure Compliance & Data Security:
Adhere to HIPAA, GDPR, and local regulations to safeguard patient information.
7. Explore Partnerships & Licensing:
Collaborate with hospitals, EMR vendors, and telemedicine providers to embed your solution in clinical workflows.
8. Expand Beyond Healthcare:
Adapt the RAG architecture for other regulated fields like law or finance where reliable info synthesis is critical.

# Actionable Insights for Prompting & Model Use

1. Prompt Length Matters:
Longer prompts increase latency. Train users to create concise queries or pre-process inputs to reduce token counts.
2. Balance Output Length & Completeness:
Limit output tokens to avoid excessive latency while maintaining answer quality.
3. Quantization & Hardware Acceleration:
Use quantized models and faster hardware to improve evaluation and generation speed.
4. Temperature Tuning by Use Case:
Medical/technical applications benefit from low temperature; creative applications may allow higher.
5. Monitor Usage Patterns & Feedback:
Collect metrics on prompt sizes, latency, and relevance to continuously optimize system performance.
6. Use Multiple Retrieval Settings & Manual Review:
Avoid relying on a single prompt or retrieval parameter. Testing with varying top_k values and prompts plus expert review prevents false or misleading outputs and supports safer clinical use.

Power Ahead